# 01 – Data Cleaning & Transformation
Here we clean the raw data, fix missing values, convert timestamps, and calculate helpfulness metrics. The goal is to prepare a clean dataset for analysis.


### Setting up the analytics schema
All cleaned and transformed tables will go into this schema.


In [ ]:
USE DATABASE AMAZON_REVIEWS_DB;
USE WAREHOUSE COMPUTE_WH;

CREATE SCHEMA IF NOT EXISTS ANALYTICS;

USE SCHEMA ANALYTICS;


### Creating the cleaned table
Transforming raw data: fixing null values, calculating helpfulness ratio, converting dates, and removing bad rows.


In [ ]:
CREATE OR REPLACE TABLE ANALYTICS.REVIEWS_CLEAN AS
SELECT
    r.ID                         AS REVIEW_ID,
    r.PRODUCTID                  AS PRODUCT_ID,
    r.USERID                     AS USER_ID,
    r.PROFILENAME                AS PROFILE_NAME,
    r.HELPFULNESSNUMERATOR       AS HELPFUL_YES,
    r.HELPFULNESSDENOMINATOR     AS HELPFUL_TOTAL,
    IFF(r.HELPFULNESSDENOMINATOR > 0,
        r.HELPFULNESSNUMERATOR / r.HELPFULNESSDENOMINATOR::FLOAT,
        0
    )                            AS HELPFUL_RATIO,
    r.SCORE                      AS RATING,
    TO_DATE(TO_TIMESTAMP_NTZ(r.TIME)) AS REVIEW_DATE,
    r.SUMMARY                    AS SUMMARY,
    r.TEXT                       AS REVIEW_TEXT
FROM RAW.REVIEWS_RAW r
WHERE r.SCORE BETWEEN 1 AND 5              
  AND r.HELPFULNESSNUMERATOR >= 0
  AND r.HELPFULNESSDENOMINATOR >= r.HELPFULNESSNUMERATOR
  AND r.TEXT IS NOT NULL;


### Quick data quality checks
Checking row counts, missing values, and basic rating stats.


In [ ]:
SELECT COUNT(*) AS ROW_COUNT
FROM ANALYTICS.REVIEWS_CLEAN;

SELECT *
FROM ANALYTICS.REVIEWS_CLEAN
LIMIT 20;

SELECT COUNT(*) AS NULL_PRODUCT_ID
FROM ANALYTICS.REVIEWS_CLEAN
WHERE PRODUCT_ID IS NULL;

SELECT RATING, COUNT(*) AS N_REVIEWS
FROM ANALYTICS.REVIEWS_CLEAN
GROUP BY RATING
ORDER BY RATING;


### Creating summary tables
These tables help with dashboards: product-level stats, user stats, and daily trends.


In [ ]:
CREATE OR REPLACE TABLE ANALYTICS.PRODUCT_METRICS AS
SELECT
    PRODUCT_ID,
    COUNT(*)                         AS N_REVIEWS,
    AVG(RATING)                      AS AVG_RATING,
    AVG(HELPFUL_RATIO)               AS AVG_HELPFUL_RATIO
FROM ANALYTICS.REVIEWS_CLEAN
GROUP BY PRODUCT_ID;


In [ ]:
CREATE OR REPLACE TABLE ANALYTICS.USER_METRICS AS
SELECT
    USER_ID,
    COUNT(*)           AS N_REVIEWS,
    AVG(RATING)        AS AVG_USER_RATING,
    AVG(HELPFUL_RATIO) AS AVG_USER_HELPFUL_RATIO
FROM ANALYTICS.REVIEWS_CLEAN
GROUP BY USER_ID;


In [ ]:
CREATE OR REPLACE TABLE ANALYTICS.DAILY_RATINGS AS
SELECT
    REVIEW_DATE,
    COUNT(*)      AS N_REVIEWS,
    AVG(RATING)   AS AVG_DAILY_RATING
FROM ANALYTICS.REVIEWS_CLEAN
GROUP BY REVIEW_DATE;


In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()

df_clean = session.table("ANALYTICS.REVIEWS_CLEAN").limit(1000).to_pandas()

print("Shape:", df_clean.shape)
print("\nColumns:", df_clean.columns.tolist())
print("\nRating summary:")
print(df_clean["RATING"].describe())
